# MOFA-FLEX model on GTEx data with Biological Process (BP) pathway priors

💡 **Environment:** `clamp-analyses`  

MOFA-FLEX model creation using GTEx gene expression data with GO Biological.

## Libraries

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
import mofaflex as mfl
from pyprojroot import here
from pathlib import Path
import pickle

/home/msubirana/miniconda3/envs/clamp-analyses/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



## Input

In [2]:
gtex_data = pd.read_csv(here("output/gtex/df_gtex_fbm_filt.csv"), index_col=0).astype(np.float32)

print(f"GTEx data shape: {gtex_data.shape}")
print(f"  Samples: {gtex_data.shape[0]}")
print(f"  Genes: {gtex_data.shape[1]}")

GTEx data shape: (21613, 17382)
  Samples: 21613
  Genes: 17382


In [3]:
K = pd.read_csv(here("output/gtex/CLAMP_K_gtex.csv"))
n_components = int(K['x'].iloc[0])

print(f"Number of components: {n_components}")

Number of components: 412


# Load BP pathway annotations

In [4]:
# get gene list from data (genes are row index)
gene_list = gtex_data.index.tolist()

# get GO BP gene sets
bp_collection = mfl.tl.msigdb_get_features(
    category="c5.go.bp",  # GO Biological Process
    dbver="7.5.1"
)

# filter to pathways that overlap with our genes
bp_collection = bp_collection.filter(
    gene_list,
    min_fraction=0.4,
    min_count=40,
    max_count=200
)

# merge similar pathways to reduce redundancy
bp_collection = bp_collection.merge_similar(
    metric="jaccard",
    similarity_threshold=0.8,
    iteratively=True,
)

print(f"Filtered BP pathways: {len(bp_collection)}")

INFO	Found 108 pairs to merge.
INFO	Found 4 pairs to merge.
INFO	Found 0 pairs to merge. Stopping...


Filtered BP pathways: 1550


## Output

In [5]:
out_dir = Path(here("output/gtex/MOFA_FLEX"))
out_dir.mkdir(parents=True, exist_ok=True)

# AnnData object

In [6]:
# transpose: AnnData expects samples as rows, genes as columns
adata = ad.AnnData(
    X=gtex_data.T.values,  # transpose to (samples x genes)
    obs=pd.DataFrame(index=gtex_data.columns),  # samples
    var=pd.DataFrame(index=gtex_data.index)     # genes
)

print(f"AnnData shape: {adata.shape}")
print(f"  Samples (obs): {adata.n_obs}")
print(f"  Genes (var): {adata.n_vars}")

# convert gene sets to binary mask (genes x pathways)
adata.varm["annotations"] = bp_collection.to_mask(gene_list).T

print(f"Annotations shape: {adata.varm['annotations'].shape}")
print(f"Number of BP pathways: {adata.varm['annotations'].shape[1]}")

AnnData shape: (17382, 21613)
  Samples (obs): 17382
  Genes (var): 21613
Annotations shape: (21613, 1550)
Number of BP pathways: 1550


# MOFA-FLEX model

In [ ]:
data_options = mfl.DataOptions(
    scale_per_group=False,        # data is already z-scored
    plot_data_overview=False,
    annotations_varm_key="annotations",
)

# model options
model_options = mfl.ModelOptions(
    n_factors=n_components,       # number of factors from K matrix
    weight_prior="Horseshoe",     # horseshoe prior required for annotations
    likelihoods="Normal",         # gaussian likelihood for continuous data
)

# training options
training_options = mfl.TrainingOptions(
    seed=42,                      # for reproducibility
    max_epochs=2000,
    save_path=False,              # don't save intermediate checkpoints
)

# train the model
model = mfl.MOFAFLEX(
    {"group_1": {"view_1": adata}},
    data_options,
    model_options,
    training_options,
)

WARNING	Device cuda is not available. Using default device: cpu
WARNING	Could not import dask. Data arrays may be copied, resulting in high memory usage.
INFO	Initializing factors using `random` method...
 20%|█▉        | 1995/10000 [15:53:23<63:45:32, 28.67s/epochs, Loss=3.65e+4]


KeyboardInterrupt: 

# Save results

In [ ]:
factors = model.get_factors()["group_1"]
# factors is (samples x LVs), so transpose to get (LVs x samples)
B_matrix = factors.T

weights = model.get_weights()["view_1"]

# save B matrix (LVs x samples)
B_matrix.to_csv(out_dir / "B_matrix.csv")

# save Z matrix (factors x genes)
weights.to_csv(out_dir / "Z_matrix.csv")

# save model as pickle
with open(out_dir / "model.pkl", "wb") as f:
    pickle.dump(model, f)
print(f"\nSaved model to {out_dir / 'model.pkl'}")